# 05 – Prediction Models

Trains and evaluates multiple ML models to predict next-day closing price:

| # | Model | Library |
|---|---|---|
| 1 | Linear Regression (baseline) | scikit-learn |
| 2 | Random Forest | scikit-learn |
| 3 | Gradient Boosting | scikit-learn |
| 4 | XGBoost | xgboost |
| 5 | LSTM | TensorFlow/Keras (optional) |

**Evaluation metrics:** MAE, RMSE, MAPE, R²

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing import load_processed
from src.analysis import TechnicalAnalysis
from src.model import StockPredictor

TICKER  = 'AAPL'
HORIZON = 1          # days ahead to predict

df = load_processed(TICKER)
df = TechnicalAnalysis(df).add_all_indicators()
df.dropna(inplace=True)

print(f'Dataset shape: {df.shape}')
df.tail(3)

## 5.1 Initialise Predictor

In [ ]:
predictor = StockPredictor(df, target_col='close', horizon=HORIZON, test_size=0.2)
print(f'Features used ({len(predictor.feature_cols)}):')
print(predictor.feature_cols)

## 5.2 Baseline – Linear Regression

In [ ]:
predictor.train_linear()
lr_metrics = predictor.evaluate()
print('Linear Regression:')
print(lr_metrics.to_string())

## 5.3 Random Forest

In [ ]:
predictor.train_random_forest(n_estimators=300)
rf_metrics = predictor.evaluate()
print('Random Forest:')
print(rf_metrics.to_string())

### Feature Importance

In [ ]:
importance = predictor.feature_importance().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
importance[::-1].plot(kind='barh', ax=ax, color='#3498db', edgecolor='none')
ax.set_title('Random Forest – Top 15 Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../images/charts/rf_feature_importance.png', dpi=150)
plt.show()

## 5.4 Gradient Boosting

In [ ]:
predictor.train_gradient_boosting(n_estimators=300, learning_rate=0.05)
gb_metrics = predictor.evaluate()
print('Gradient Boosting:')
print(gb_metrics.to_string())

## 5.5 XGBoost

In [ ]:
try:
    predictor.train_xgboost()
    xgb_metrics = predictor.evaluate()
    print('XGBoost:')
    print(xgb_metrics.to_string())
except ImportError as e:
    print(f'Skipping XGBoost: {e}')
    xgb_metrics = None

## 5.6 LSTM (optional – requires TensorFlow)

In [ ]:
try:
    predictor.train_lstm(lookback=60, epochs=30, batch_size=32)
    lstm_metrics = predictor.evaluate()
    print('LSTM:')
    print(lstm_metrics.to_string())
except ImportError as e:
    print(f'Skipping LSTM: {e}')
    lstm_metrics = None

## 5.7 Model Comparison

In [ ]:
comparison = {
    'Linear Regression': lr_metrics,
    'Random Forest':     rf_metrics,
    'Gradient Boosting': gb_metrics,
}
if xgb_metrics  is not None: comparison['XGBoost'] = xgb_metrics
if lstm_metrics is not None: comparison['LSTM']    = lstm_metrics

comparison_df = pd.DataFrame(comparison).T
print('\n📊 Model Comparison:')
comparison_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    vals = comparison_df[metric]
    colors = ['#26a69a' if v == vals.min() else '#546e7a' for v in vals]
    if metric == 'R²':
        colors = ['#26a69a' if v == vals.max() else '#546e7a' for v in vals]
    ax.barh(vals.index, vals.values, color=colors, edgecolor='none')
    ax.set_title(metric, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/charts/model_comparison.png', dpi=150)
plt.show()

## 5.8 Actual vs Predicted (Best Model)

In [ ]:
# Re-train best model (Random Forest) and plot predictions vs actuals
predictor.train_random_forest(n_estimators=300)
y_pred = predictor.model.predict(predictor.X_test)
y_true = predictor.y_test

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_true,  label='Actual',    color='#ecf0f1', linewidth=1.2)
ax.plot(y_pred,  label='Predicted', color='#26a69a', linewidth=1.2, linestyle='--')
ax.set_title(f'{TICKER} – Actual vs Predicted (Random Forest, T+{HORIZON})', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/charts/actual_vs_predicted.png', dpi=150)
plt.show()

---
✅ **Analysis complete.** See `reports/insights_summary.md` for the summary of findings.